# 부록 B — 온톨로지 정의 생성기 (generate_definition)

계약을 읽어 `definition_parts/`를 생성합니다.

## 1 · Import & 설정

In [ ]:
# 계약(ontology_contract.yaml) → Fabric Ontology definition parts(JSON) 생성기 (Notebook 판)
import json
import shutil
import uuid
from pathlib import Path

import yaml

CONTRACT = "ontology_contract.yaml"
OUT_DIR = "definition_parts"
BINDING_WORKSPACE_ID = "00000000-0000-0000-0000-000000000000"
BINDING_ITEM_ID = "00000000-0000-0000-0000-000000000000"
BINDING_SOURCE_SCHEMA = "dbo"

_NAMESPACE = uuid.UUID("6f1e2d3c-4b5a-6978-8a9b-0c1d2e3f4a5b")
UUID_ZERO = "00000000-0000-0000-0000-000000000000"

SCHEMA_ENTITY = "https://developer.microsoft.com/json-schemas/fabric/item/ontology/entityType/1.0.0/schema.json"
SCHEMA_DATABINDING = "https://developer.microsoft.com/json-schemas/fabric/item/ontology/dataBinding/1.0.0/schema.json"
SCHEMA_RELATIONSHIP = "https://developer.microsoft.com/json-schemas/fabric/item/ontology/relationshipType/1.0.0/schema.json"
SCHEMA_CONTEXTUALIZATION = "https://developer.microsoft.com/json-schemas/fabric/item/ontology/contextualization/1.0.0/schema.json"
SCHEMA_PLATFORM = "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json"

## 2 · 헬퍼 함수

In [ ]:
def det_uuid(*parts: str) -> str:
    """계약 상의 이름들로부터 결정적 UUID를 생성한다."""
    return str(uuid.uuid5(_NAMESPACE, "::".join(parts)))

def infer_value_type(column: str) -> str:
    """컬럼명 규칙으로 Fabric Ontology valueType 을 추론한다."""
    col = column.lower()
    if col.endswith("_at"):
        return "DateTime"
    if col.endswith("_date") or col == "join_date":
        return "DateTime"
    if col in {"quantity", "on_hand_qty", "reserved_qty"} or col.endswith("_qty"):
        return "BigInt"
    if (
        col.endswith("_amount")
        or col.endswith("_price")
        or col.endswith("_value")
        or col.endswith("_revenue")
        or col in {"discount_applied", "unit_price"}
    ):
        return "Double"
    return "String"

## 3 · 빌더 함수

In [ ]:
def build_entities(contract: dict, binding_workspace_id: str, binding_item_id: str, binding_source_schema: str) -> tuple[dict, dict, list]:
    """엔터티 정의/데이터바인딩 parts 와, 관계 생성에 필요한 인덱스를 만든다."""
    entity_ids: dict[str, str] = {}
    property_ids: dict[str, dict[str, str]] = {}
    parts: list[dict] = []

    for entity_name, spec in contract["entities"].items():
        entity_id = det_uuid("entity", entity_name)
        entity_ids[entity_name] = entity_id
        property_ids[entity_name] = {}

        # ---- Property (엔터티 정의 안에 인라인) ----
        properties = []
        for column in spec["fields"]:
            pid = det_uuid("property", entity_name, column)
            property_ids[entity_name][column] = pid
            properties.append(
                {
                    "id": pid,
                    "name": column,
                    "valueType": infer_value_type(column),
                    "sourceColumnName": column,
                }
            )

        pk_cols = spec["primary_key"]
        if isinstance(pk_cols, str):
            pk_cols = [pk_cols]
        entity_id_parts = [property_ids[entity_name][c] for c in pk_cols]
        display_col = spec.get("display", pk_cols[0])
        display_property_id = property_ids[entity_name][display_col]

        entity_definition = {
            "$schema": SCHEMA_ENTITY,
            "id": entity_id,
            "namespace": "usertypes",
            "baseEntityTypeId": None,
            "name": entity_name,
            "description": f"{entity_name} (source: {spec['source']})",
            "entityIdParts": entity_id_parts,
            "displayNamePropertyId": display_property_id,
            "namespaceType": "Custom",
            "visibility": "Visible",
            "properties": properties,
            "timeseriesProperties": [],
            "untypedProperties": [],
        }
        parts.append(
            {
                "path": f"EntityTypes/{entity_id}/definition.json",
                "content": entity_definition,
            }
        )

        # ---- DataBinding (Property ↔ Lakehouse 컬럼 매핑) ----
        binding_id = det_uuid("databinding", entity_name)
        property_bindings = [
            {
                "sourceColumnName": column,
                "targetPropertyId": property_ids[entity_name][column],
            }
            for column in spec["fields"]
        ]
        data_binding = {
            "$schema": SCHEMA_DATABINDING,
            "id": binding_id,
            "dataBindingConfiguration": {
                "dataBindingType": "NonTimeSeries",
                "propertyBindings": property_bindings,
                "sourceTableProperties": {
                    "sourceType": "LakehouseTable",
                    "workspaceId": binding_workspace_id,
                    "itemId": binding_item_id,
                    "sourceTableName": spec["source"],
                    "sourceSchema": binding_source_schema,
                },
            },
        }
        parts.append(
            {
                "path": f"EntityTypes/{entity_id}/DataBindings/{binding_id}.json",
                "content": data_binding,
            }
        )

    return entity_ids, property_ids, parts

def build_relationships(
    contract: dict,
    entity_ids: dict,
    property_ids: dict,
    binding_workspace_id: str,
    binding_item_id: str,
    binding_source_schema: str,
) -> list:
    """물리 관계(Contextualization 포함) + 논리 관계(정의만) parts 를 만든다."""
    parts: list[dict] = []

    # ---- 물리 관계: 관계 정의 + Contextualization(FK 바인딩) ----
    for rel in contract.get("relationships", []):
        rel_id = det_uuid("relationship", rel["name"])
        from_e, to_e = rel["from_entity"], rel["to_entity"]

        relationship_definition = {
            "$schema": SCHEMA_RELATIONSHIP,
            "id": rel_id,
            "namespace": "usertypes",
            "name": rel["name"],
            "displayName": rel.get("display_name", rel["name"]),
            "description": f"{rel.get('cardinality', '')} ({from_e} → {to_e})",
            "namespaceType": "Custom",
            "source": {"entityTypeId": entity_ids[from_e]},
            "target": {"entityTypeId": entity_ids[to_e]},
        }
        parts.append(
            {
                "path": f"RelationshipTypes/{rel_id}/definition.json",
                "content": relationship_definition,
            }
        )

        ctx_id = det_uuid("contextualization", rel["name"])
        contextualization = {
            "$schema": SCHEMA_CONTEXTUALIZATION,
            "id": ctx_id,
            "relationshipTypeId": rel_id,
            "dataBindingTable": {
                "sourceType": "LakehouseTable",
                "workspaceId": binding_workspace_id,
                "itemId": binding_item_id,
                "sourceTableName": rel["binding_table"],
                "sourceSchema": binding_source_schema,
            },
            "sourceKeyRefBindings": [
                {
                    "sourceColumnName": rel["from_key"],
                    "targetPropertyId": property_ids[from_e].get(rel["from_key"]),
                }
            ],
            "targetKeyRefBindings": [
                {
                    "sourceColumnName": rel["to_key"],
                    "targetPropertyId": property_ids[to_e].get(rel["to_key"]),
                }
            ],
        }
        parts.append(
            {
                "path": f"RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json",
                "content": contextualization,
            }
        )

    # ---- 논리 관계: 관계 정의만 (FK 없음) ----
    for rel in contract.get("logical_relationships", []):
        rel_id = det_uuid("relationship", rel["name"])
        from_e, to_e = rel["from_entity"], rel["to_entity"]
        relationship_definition = {
            "$schema": SCHEMA_RELATIONSHIP,
            "id": rel_id,
            "namespace": "usertypes",
            "name": rel["name"],
            "displayName": rel.get("display_name", rel["name"]),
            "description": f"LOGICAL {rel.get('cardinality', '')} derived via {rel.get('derived_via', '')}",
            "isLogical": True,
            "namespaceType": "Custom",
            "source": {"entityTypeId": entity_ids[from_e]},
            "target": {"entityTypeId": entity_ids[to_e]},
        }
        parts.append(
            {
                "path": f"RelationshipTypes/{rel_id}/definition.json",
                "content": relationship_definition,
            }
        )

    return parts

def build_root_parts(contract: dict) -> list:
    """definition.json(빈) 과 .platform(아이템 메타) parts."""
    onto = contract["ontology"]
    platform = {
        "$schema": SCHEMA_PLATFORM,
        "metadata": {"type": "Ontology", "displayName": onto["display_name"]},
        "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"},
    }
    return [
        {"path": "definition.json", "content": {}},
        {"path": ".platform", "content": platform},
    ]

## 4 · 실행

In [ ]:
with open(CONTRACT, "r", encoding="utf-8") as fh:
    contract = yaml.safe_load(fh)

entity_ids, property_ids, entity_parts = build_entities(
    contract,
    BINDING_WORKSPACE_ID,
    BINDING_ITEM_ID,
    BINDING_SOURCE_SCHEMA,
)
relationship_parts = build_relationships(
    contract,
    entity_ids,
    property_ids,
    BINDING_WORKSPACE_ID,
    BINDING_ITEM_ID,
    BINDING_SOURCE_SCHEMA,
)
root_parts = build_root_parts(contract)
all_parts = root_parts + entity_parts + relationship_parts

out_dir = Path(OUT_DIR)
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

manifest_parts = []
for part in all_parts:
    target = out_dir / part["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    with open(target, "w", encoding="utf-8") as fh:
        json.dump(part["content"], fh, ensure_ascii=False, indent=2)
    manifest_parts.append(part["path"])

manifest = {
    "ontologyDisplayName": contract["ontology"]["display_name"],
    "description": contract["ontology"]["description"],
    "counts": {
        "entities": len(contract["entities"]),
        "physicalRelationships": len(contract.get("relationships", [])),
        "logicalRelationships": len(contract.get("logical_relationships", [])),
        "totalParts": len(all_parts),
    },
    "entityIds": entity_ids,
    "parts": manifest_parts,
}
with open(out_dir / "_manifest.json", "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

print(f"✅ 생성 완료: {out_dir.resolve()}")
print(json.dumps(manifest["counts"], ensure_ascii=False, indent=2))